<a href="https://colab.research.google.com/github/Raidum-22/1ms25is021_assignment2/blob/main/movie_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [ ]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
MOVIE_DATABASE = {
   "sci-fi": [
        {"title": "Interstellar", "year": 2014, "rating": 8.7, "desc": "A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival."},
        {"title": "Blade Runner 2049", "year": 2017, "rating": 8.0, "desc": "A young Blade Runner's discovery of a long-buried secret leads him to track down former Blade Runner Rick Deckard."},
        {"title": "Inception", "year": 2010, "rating": 8.8, "desc": "A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea."}
    ],
    "action": [
        {"title": "Mad Max: Fury Road", "year": 2015, "rating": 8.1, "desc": "In a post-apocalyptic wasteland, a woman mebers against a tyrannical ruler in search for her homeland."},
        {"title": "John Wick", "year": 2014, "rating": 7.4, "desc": "An ex-hit-man comes out of retirement to track down the gangsters that killed his dog."}
    ],
    "comedy": [
        {"title": "Superbad", "year": 2007, "rating": 7.6, "desc": "Two co-dependent high school seniors deal with separation anxiety after their plan to host a booze-soaked party goes awry."},
        {"title": "The Grand Budapest Hotel", "year": 2014, "rating": 8.1, "desc": "A writer encounters the owner of a high-class European hotel who tells him of his early years as a lobby boy."}
    ]
}

In [ ]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, movies in MOVIE_DATABASE.items():
        words = set(query_lower.split())
        # Search in movie titles, descriptions, and the genre name itself
        for movie in movies:
            combined_text = f"{movie['title']} {movie['desc']} {doc_name}".lower()
            match_count = sum(1 for word in words if word in combined_text)
            if match_count > 0:
                results.append((match_count, doc_name, movie))

    results.sort(reverse=True, key=lambda x: x[0])

    if not results:
        return "No relevant movies found for that query."

    # Return the top two matches
    output = []
    for _, name, movie in results[:2]:
        output.append(f"[Source: {name}]\nTitle: {movie['title']} ({movie['year']}) - Rating: {movie['rating']}\nDescription: {movie['desc']}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("Action"))

[Source: action]
Title: Mad Max: Fury Road (2015) - Rating: 8.1
Description: In a post-apocalyptic wasteland, a woman mebers against a tyrannical ruler in search for her homeland.

[Source: action]
Title: John Wick (2014) - Rating: 7.4
Description: An ex-hit-man comes out of retirement to track down the gangsters that killed his dog.


In [ ]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        """
        You are a helpful and knowledgeable Movie Recommendation Agent.
Your job is to recommend movies based on genres specified by the user.
Always use the `search_documents` tool when a user asks for recommendations in a specific genre.
Present your final recommendations clearly with movie title, year, rating, and a brief description.

CRITICAL RULE:
If a genre requested by the user is unavailable in the database (such as Horror), you MUST NOT suggest external movies or ask follow-up questions about that unavailable genre. Instead, explicitly state that the requested genre is not available, and suggest that they choose from one of our available genres: Sci-Fi, Action, or Comedy.
Only answer general questions when not pertaining specifically to recommendations.
"""
    )
)

print("Model ready:", model.model_name)

/tmp/ipykernel_2850/1820917290.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Model ready: models/gemini-3.5-flash-lite


In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant movie to recommend.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("Can you recommend some good Sci-Fi movies to watch tonight?")

Question: Can you recommend some good Sci-Fi movies to watch tonight?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'sci-fi movies'})
  Result preview: [Source: sci-fi]
Title: Interstellar (2014) - Rating: 8.7
Description: A team of explorers travel th...

Final answer:
Here are a couple of great sci-fi movie recommendations for tonight:

*   **Interstellar (2014) - Rating: 8.7**
    *   *Description:* A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.
*   **Blade Runner 2049 (2017) - Rating: 8.0**
    *   *Description:* A young Blade Runner's discovery of a long-buried secret leads him to track down former Blade Runner Rick Deckard.

Enjoy your movie night! Let me know if you would like more suggestions in a specific sub-genre (like space travel, cyberpunk, or thrillers).


In [10]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What about Action movies instead?",
    "How about a light-hearted comedy movie?",
    # This question is not answered by the database.
    "Can you recommend a good Horror movie?",
    # This question is part of the general knowledge of the model.
    "Who directed the movie Titanic, and what was its production budget?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What about Action movies instead?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Action'})
  Result preview: [Source: action]
Title: Mad Max: Fury Road (2015) - Rating: 8.1
Description: In a post-apocalyptic w...

Final answer:
Here are a couple of great Action movie recommendations:

* **Mad Max: Fury Road (2015)** - Rating: 8.1
  * *Description:* In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler in search of her homeland.
* **John Wick (2014)** - Rating: 7.4
  * *Description:* An ex-hit-man comes out of retirement to track down the gangsters that killed his dog. 

Would you like to know more about either of these, or are you looking for a specific style of action movie?

Question: How about a light-hearted comedy movie?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'light-hearted comedy'})
  Result preview: [Source: comedy]
Title: Superbad (2007)

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)